In [ ]:
import json

import numpy as np
import scipy.io as sio

file_name = "fields_002000.mat"    # or a .json/.yaml SCFT result

# Read an SCFT result (.mat, .json or .yaml) or an L-FTS or CL-FTS fields file (.mat)
if file_name.endswith(".json"):
    with open(file_name) as f:
        data = json.load(f)
elif file_name.endswith((".yaml", ".yml")):
    import yaml
    with open(file_name) as f:
        data = yaml.safe_load(f)
else:
    data = sio.loadmat(file_name, simplify_cells=True)     # MATLAB structures as dicts
nx = np.atleast_1d(np.asarray(data['nx'], dtype=int))
lx = np.atleast_1d(np.asarray(data['lx'], dtype=float))

# The angles of the cell in degrees: saved in the file, or in the checkpoint settings or the input
# of older files ([gamma] in 2D)
angles = data.get('angles')
if angles is None and 'checkpoint_settings_json' in data:
    angles = json.loads(str(data['checkpoint_settings_json']))['angles']
if angles is None:
    angles = data.get('initial_params', {}).get('angles')
angles = np.atleast_1d(np.asarray([] if angles is None else angles, dtype=float))
if angles.size == 0:
    print("The file holds no lattice angles: the cell is drawn with right angles.")
    angles = np.full(3, 90.0)
elif angles.size == 1:
    angles = np.array([90.0, 90.0, angles[0]])

# The grid points sit at (i + offset)/n along each lattice vector, as in the solvers: the cell
# centres, or the nodes along a and b for a space group on hexagonal axes. An SCFT result with a
# space group saves its Hall number: the trigonal and hexagonal settings 430-488 use hexagonal
# axes, except the R settings on rhombohedral axes (SpaceGroup.uses_hexagonal_axes)
hexagonal = False
if 'space_group_hall_number' in data:
    hall = int(data['space_group_hall_number'])
    hexagonal = 430 <= hall <= 488 and hall not in (434, 437, 445, 451, 453, 459, 461)
offsets = np.full(len(nx), 0.5)
if hexagonal:
    offsets[:2] = 0.0

# phi_<type>, the real part phi_<type>_real of CL-FTS, or the phi structure of older files
older = data.get('phi', {})
if 'monomer_types' in data:
    names = [str(name).strip() for name in np.atleast_1d(data['monomer_types'])]
elif older:
    names = list(older)
else:
    names = [key[4:].removesuffix('_real') for key in data if key.startswith('phi_') and not key.endswith('_imag')]
phi = {}
for name in names:
    value = data.get('phi_' + name, data.get('phi_' + name + '_real', older.get(name)))
    if value is None:
        raise KeyError(f"The file holds no concentration of monomer type {name}")
    phi[name] = np.asarray(value).reshape(nx)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

slice_index = 0     # the plane of grid index slice_index along the lattice vector a, 0 to nx[0] - 1

if not 0 <= slice_index < nx[0]:
    raise ValueError(f"slice_index must be 0 to {nx[0] - 1}, got {slice_index}")
for name, values in phi.items():
    print(f"phi_{name} min: {values.min():.4f}, max: {values.max():.4f}")

# b along x, c in the xy plane at the angle alpha
alpha = np.deg2rad(angles[0])
b = lx[1]*np.array([1.0, 0.0])
c = lx[2]*np.array([np.cos(alpha), np.sin(alpha)])

# Corners of the grid cells: each value fills the cell around its point, (j + offset)/n along b
# and c as in the solvers (the nodes along b on hexagonal axes)
fb, fc = np.meshgrid(*[(np.arange(n + 1) + o - 0.5)/n for n, o in zip(nx[1:], offsets[1:])], indexing='ij')
X, Y = fb*b[0] + fc*c[0], fb*b[1] + fc*c[1]

# Every monomer type of the file on the plane: [j, k], along b and along c
fig, axes = plt.subplots(1, len(phi), figsize=(5*len(phi), 5))
axes = np.atleast_1d(axes)
for ax, (name, values) in zip(axes, phi.items()):
    im = ax.pcolormesh(X, Y, values[slice_index], cmap=cm.jet, vmin=0.0, vmax=1.0)
    ax.set(title=f'phi_{name}, plane {slice_index} along a', xlabel='x', ylabel='y', aspect='equal')
fig.colorbar(im, ax=axes.tolist())
fig.show()

# fig.savefig("2d_slice_image.png", dpi=300, bbox_inches='tight', pad_inches=0)